In [0]:
from pyspark.sql.functions import col,sum,count,avg,round

spark.sql("create catalog if not exists sales_catalog")
spark.sql("use catalog sales_catalog")
spark.sql("create schema if not exists star_schema")
spark.sql("use SCHEMA  star_schema")


In [0]:
# Dim Customer
dim_customer = spark.createDataFrame([
    (1, "Rahul Sharma", "Delhi", "North", "Premium"),
    (2, "Priya Singh", "Mumbai", "West", "Regular"),
    (3, "Amit Kumar", "Bangalore", "South", "Premium"),
    (4, "Sneha Patel", "Chennai", "South", "Regular"),
    (5, "Vikram Mehta", "Delhi", "North", "Premium")
], ["customer_id", "customer_name", "city", "region", "segment"])

# Dim Product
dim_product = spark.createDataFrame([
    (1, "Laptop", "Electronics", "HP", 75000),
    (2, "Phone", "Electronics", "Samsung", 25000),
    (3, "Shirt", "Clothing", "Allen Solly", 1500),
    (4, "Shoes", "Clothing", "Nike", 5000),
    (5, "Tablet", "Electronics", "Apple", 45000)
], ["product_id", "product_name", "category", "brand", "unit_price"])

# Dim Date
dim_date = spark.createDataFrame([
    (1, "2024-01-15", 2024, 1, "Q1", "January", "Monday"),
    (2, "2024-02-20", 2024, 2, "Q1", "February", "Tuesday"),
    (3, "2024-03-10", 2024, 3, "Q1", "March", "Sunday"),
    (4, "2024-04-05", 2024, 4, "Q2", "April", "Friday"),
    (5, "2024-05-15", 2024, 5, "Q2", "May", "Wednesday")
], ["date_id", "full_date", "year", "month", "quarter", "month_name", "day_name"])

# Dim Region
dim_region = spark.createDataFrame([
    (1, "North", "Delhi", "India"),
    (2, "South", "Bangalore", "India"),
    (3, "West", "Mumbai", "India"),
    (4, "East", "Kolkata", "India")
], ["region_id", "region_name", "main_city", "country"])

# Save all Dim tables
dim_customer.write.format("delta").mode("overwrite").saveAsTable("sales_catalog.star_schema.dim_customer")
dim_product.write.format("delta").mode("overwrite").saveAsTable("sales_catalog.star_schema.dim_product")
dim_date.write.format("delta").mode("overwrite").saveAsTable("sales_catalog.star_schema.dim_date")
dim_region.write.format("delta").mode("overwrite").saveAsTable("sales_catalog.star_schema.dim_region")

print("All Dimension Tables Created!")
print("- dim_customer ")
print("- dim_product ")
print("- dim_date ")
print("- dim_region ")

In [0]:
# Fact Sales Table
fact_sales = spark.createDataFrame([
    (1, 1, 1, 1, 1, 2, 150000),
    (2, 2, 2, 2, 2, 1, 25000),
    (3, 3, 3, 3, 3, 3, 4500),
    (4, 4, 1, 4, 1, 1, 75000),
    (5, 5, 2, 5, 3, 2, 90000),
    (6, 1, 3, 1, 2, 1, 45000),
    (7, 2, 4, 2, 4, 2, 10000),
    (8, 3, 5, 3, 1, 3, 225000),
    (9, 4, 1, 4, 5, 1, 25000),
    (10, 5, 2, 5, 2, 2, 50000)
], ["sale_id", "customer_id", "product_id", "date_id", 
    "region_id", "quantity", "total_amount"])

fact_sales.write.format("delta").mode("overwrite").saveAsTable("sales_catalog.star_schema.fact_sales")

print(" Fact Table Created!")
print(f"Total sales records: {fact_sales.count()}")
fact_sales.display()

**Star Schema Queries**

In [0]:
#Query 1 -Revenue by Customer Segment
print("Revenue by Customer Segment")

spark.sql("""
          
          select c.segment,
          count(f.sale_id) as total_orders,
          sum(f.total_amount) as total_revenue,
          round(avg(f.total_amount),2) as avg_order_value from sales_catalog.star_schema.fact_sales f
          join sales_catalog.star_schema.dim_customer c
          on f.customer_id=c.customer_id
          group by c.segment
          order by total_revenue desc
          """).show()

In [0]:
#Query 2-Revenue by Product Category
print("Revenue by Product Category")
spark.sql("""
          select p.category,
            p.brand,
            sum(f.quantity) as units_sold,
            sum(f.total_amount) as total_revenue
          from sales_catalog.star_schema.fact_sales f
           join sales_catalog.star_schema.dim_product p
          on f.product_id=p.product_id
          group by p.category,p.brand
          order by total_revenue desc
          """).display()

In [0]:
# Query 3 - Revenue by Region and Quarter
print("📊 Revenue by Region and Quarter:")
spark.sql("""
           select r.region_name,
            d.quarter,
            sum(f.total_amount) as total_revenue
FROM sales_catalog.star_schema.fact_sales f
JOIN sales_catalog.star_schema.dim_region r 
        ON f.region_id = r.region_id
JOIN sales_catalog.star_schema.dim_date d 
        ON f.date_id = d.date_id
group by r.region_name,d.quarter
order by total_revenue desc
""").display()

In [0]:
print(" Star Schema Queries Complete!")